# Kafi Streams: Complex Stream Processing Made Simple

### <i>Ralph Debusmann (Migros-Genossenschafts-Bund)</i>

## A little bit of history

Started studying AI in 1995, PhD in 2006...

<img src="xdg.png" style="width: 30%; height: 30%"/>

AI jobs in 2006? Not really. So - went to mundane software development in 2008...

<img src="sap.png" style="width: 30%; height: 30%"/>

6 years in (2014), found out about Scala, distributed systems, Spark, and... Kafka...

<img src="kafka.png" style="width: 20%; height: 20%"/>

The heureka moment: Kafka Streams (2016)... at the time, I thought stream processing would be omnipresent in a few years time...

<img src="kafka_streams.jpg" style="width: 40%; height: 40%"/>

But then - nothing really happened. Why? I took up research again in 2023, leading up to co-authoring a book with Hubert Dulay:

<img src="streaming_databases.jpg" style="width: 30%; height: 30%"/>

At the end of writing this, I began being able to pinpoint why stream processing has never been able to really take off, writing a blog post which kind of went viral at Confluent:

<img src="mainstream.png" style="width: 30%; height: 30%"/>

So what is it that has held off stream processing? I think it's this:

<img src="leaky_abstractions.png" style="width: 30%; height: 30%"/>


## Leaky Abstractions in Stream Processing

### Imagine...

Imagine you would wish to build a stream processing pipeline.

Maybe just for joining a few topics before you ingest them into your database or search engine.

Or because you really have a low-latency, near real-time use case and you haven't touched stream processing before.

Should be easy, no?

You start using one of the most popular stream processing tools, e.g. Kafka Streams or Flink.

You get it to work.

And then you want to go to production.

<img src="boom.jpg" style="width: 30%; height: 30%"/>

### Stream Processing ≠ Data Engineering

Regardless of whether you use SQL (e.g. ksqlDB on top of Kafka Streams or FlinkSQL on top of Flink) or a lower-level API (Kafka Streams DSL, Flink DataStream API), bringing stream processing to production is still incredibly hard.

Data Engineering skills are not enough. You need to be an expert in infrastructure and distributed systems as well.

Here are some of the concepts that you have to understand in order to build a stream processing pipeline in production:

Kafka Streams:
* co-partitioning
* keying
* KStream vs. KTable vs. GlobalKTable
* join semantics (Florian Troßbach: https://www.confluent.io/blog/crossing-streams-joins-apache-kafka/)
* suppress
* grace period
* tombstone handling

Flink:
* watermarks
* mini-batch
* multi-join
* allowed lateness
* side outputs for late data
* idleness detection
* checkpoint barriers
* barrier alignment/backpressure
* window trigger/evictions
* broadcast state/joins
* managed memory fractions
* keyed stream transformation
* state TTL
* savepoints vs checkpoints
* changelog mode tuning
* sink tuning (append-only/upsert)

And, of course, a couple of concepts for both:

* eventual consistency
* exactly-once semantics
* RocksDB tuning

### So What Happens?

(Too) often, stream processing projects are either stopped or not started at all. Just too hard. Just too expensive.

<img src="stop.jpg" style="width: 20%; height: 20%"/>


## A Way Out?

During the research Hubert Dulay and me did for the O'Reilly book, we also talked with those behind Materialize and Feldera.

Materialize is based on Differential Dataflow (DD), a theory dating back to 2013 (Frank McSherry et al.):

<img src="dd.png" style="width: 40%; height: 40%"/>

Feldera is based on DataBase Stream Processing (DBSP), a simplified theory inspired by Differential Dataflow (Mihai Budiu et al. 2022):

<img src="dbsp.png" style="width: 40%; height: 40%"/>

Materialize and Feldera take a completely different approach as "classical" stream processing. Not ad-hoc. Not based on append-only streams. Not based on "infinite lists". But on *sets* and *relational algebra*.

We focus on DBSP onward.

## A Classic Example - Word Count ;-)

* "classical" stream processing:
Source topic 1, Source topic 2... Source topic N -> Stream Processor (e.g. Kafka Streams, Flink) -> Sink topic 1, Sink topic 
2, ..., Sink topic N
["hello kafi streams", "all streams lead to kafi", "join berlin buzzwords"...] -> ["hello", "kafi", "streams", "all", "streams", "lead", "to", "kafi", "join", "berlin", "buzzwords"...] -> [{"word": "hello", "count": 1}, {"word": "kafi", "count": 2}, {"word": "streams", "count": 2}, {"word": "all", "count": 1}, {"word": "hello", "count": 1}, {"word": "lead", "count": 1}, {"word": "to", "count": 1}, {"word": "join", "count": 1}, {"word": "berlin", "count": 1}, {"word": "buzzwords", "count": 1}]

* DBSP-based stream processing:
["hello kafi streams", "all streams lead to kafi", "join berlin buzzwords"...] -> [{"hello": 1, "kafi": 1, "streams": , "all", "streams", "lead", "to", "kafi", "join", "berlin", "buzzwords"...] -> [{"word": "hello", "count": 1}, {"word": "kafi", "count": 2}, {"word": "streams", "count": 2}, {"word": "all", "count": 1}, {"word": "hello", "count": 1}, {"word": "lead", "count": 1}, {"word": "to", "count": 1}, {"word": "join", "count": 1}, {"word": "berlin", "count": 1}, {"word": "buzzwords", "count": 1}]



Streaming beyond the hype
=> streaming vs batch - question just "how fresh shall the data be"
implicit: go to streaming only when data needs to be really fresh + streaming is more expensive
but: I'd say: go to streaming if you can save costs!
How is that supposed to work?

scalability - implicit: horizontal - but is this needed anymore (multicore!)



Session Abstract
You can finally stop caring about co-partitioning, state stores and eventual consistency. Kafi Streams, built on (Py)DBSP, treats streaming like batch — strongly consistent, no special concepts. An Open Source Python library for the 80% of use cases that don’t need extreme scale. Fully incremental stream processing for everyone, from day one.

Session Description
I will unveil Kafi Streams, an Open Source library for complex stream processing inspired by Kafka Streams but built on top of PyDBSP, a pure Python implementation of Feldera’s novel “Database Stream Processing” theory.

Why would we need yet another stream processing library? One whose name sounds so strikingly similar to the most popular stream processing library on the planet?

Because existing stream processing libraries are too complex, even Kafka Streams. Their engines have been prematurely optimized for maximum scale, not simplicity. You cannot easily do stream processing without understanding concepts like streams vs. tables, co-partitioning, windowing (hopping, tumbing, sessions…), state stores etc. – all these “leaky abstractions” still prevailing in the stream processing world. It is them that keep stream processing in a niche.

On the contrary, Kafi Streams aims at making stream processing simple. It does not (yet) aim for extreme scale and performance. But to enable complex stream processing with full support for joins, aggregations et al. for the less performance-heavy 80% of use cases in non-tech companies like mine, Migros, a $30B+ revenue retailer.

With Kafi Streams, anyone can do complex stream processing, even those who have never done it before. Right from the start. Because with DBSP as our basis, streaming is no different from batch any longer. Simple. Deterministic. Not just eventually but strongly consistent. Just like anybody coming from outside the streaming world would always have hoped.

